# CNN и Autoencoders

### Кратко о решении
Объединены оба задания в один ноутбук:
1. **Transfer Learning:** классификация Fashion-MNIST через MobileNetV2.
2. **Autoencoders:** удаление шума с MNIST и сравнение 2 архитектур.


## Часть 1 — Классификация изображений
**Методы:** data augmentation, MobileNetV2, frozen vs unfrozen layers, accuracy/loss comparison.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import time

print("=== ЗАДАНИЕ №1: КЛАССИФИКАЦИЯ ИЗОБРАЖЕНИЙ ===")

# 1. ВЫБОР ДАТАСЕТА - Fashion-MNIST (10 классов)
print("1. Загрузка датасета Fashion-MNIST...")
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# СОКРАЩЕНИЕ ДО 10% ДАННЫХ
print("Сокращение датасета до 10%...")
train_size = len(x_train) // 10
test_size = len(x_test) // 10

x_train = x_train[:train_size]
y_train = y_train[:train_size]
x_test = x_test[:test_size]
y_test = y_test[:test_size]

print(f"После сокращения:")
print(f"Тренировочная выборка: {x_train.shape}")
print(f"Тестовая выборка: {x_test.shape}")

# Названия классов Fashion-MNIST
class_names = ['Футболка', 'Брюки', 'Свитер', 'Платье', 'Пальто',
               'Сандали', 'Рубашка', 'Кроссовки', 'Сумка', 'Ботильоны']

# 2. ПОДГОТОВКА ДАННЫХ
print("\n2. Подготовка данных...")

# Преобразование в 3 канала (для совместимости с MobileNetV2)
x_train = np.stack([x_train] * 3, axis=-1)
x_test = np.stack([x_test] * 3, axis=-1)

# Нормализация
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# One-hot encoding
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

# Параметры
BATCH_SIZE = 32
IMG_SIZE = (128, 128)

# 2.1 АУГМЕНТАЦИЯ ДАННЫХ - УПРОЩАЕМ
print("2.1 Создание аугментации данных...")
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.05),  # Уменьшаем вращение
    keras.layers.RandomZoom(0.05),      # Уменьшаем зум
])

# Функция для предобработки с аугментацией
def preprocess_train(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = data_augmentation(image)
    return image, label

def preprocess_test(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

# Создание tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# Применяем аугментацию только к тренировочным данным
train_dataset = train_dataset.map(preprocess_train)
test_dataset = test_dataset.map(preprocess_test)

# Бэтчинг и оптимизация
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Визуализация данных после аугментации
print("Визуализация аугментированных данных...")
plt.figure(figsize=(12, 6))
for images, labels in train_dataset.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i])
        plt.axis('off')
        class_idx = np.argmax(labels[i])
        plt.title(f'{class_names[class_idx]}')
plt.suptitle('Аугментированные изображения Fashion-MNIST (10% данных)')
plt.tight_layout()
plt.show()

# 3. ИМПОРТ И АДАПТАЦИЯ ПРЕДОБУЧЕННОЙ СЕТИ
print("\n3. Создание модели на основе MobileNetV2...")

IMG_SHAPE = IMG_SIZE + (3,)

# Вариант 1: С ЗАМОРОЗКОЙ (Feature Extraction)
def create_model_with_frozen_layers():
    # Базовая модель
    base_model = keras.applications.MobileNetV2(
        input_shape=IMG_SHAPE,
        include_top=False,
        weights='imagenet'
    )
    
    # ЗАМОРОЗКА всех слоев
    base_model.trainable = False
    
    # Построение модели
    inputs = keras.Input(shape=IMG_SHAPE)
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=True)  
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation='relu')(x)  # Добавляем плотный слой
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(10, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model, base_model

# Вариант 2: БЕЗ ЗАМОРОЗКИ (Full Training)
def create_model_unfrozen():
    base_model = keras.applications.MobileNetV2(
        input_shape=IMG_SHAPE,
        include_top=False,
        weights='imagenet'
    )
    
    # НЕТ ЗАМОРОЗКИ - все слои обучаемые
    base_model.trainable = True
    
    inputs = keras.Input(shape=IMG_SHAPE)
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=True)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(10, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model, base_model

# 4. ОБУЧЕНИЕ С ЗАМОРОЗКОЙ
print("\n4.1 Обучение с ЗАМОРОЖЕННЫМИ слоями...")

model_frozen, base_model_frozen = create_model_with_frozen_layers()
print("Модель с замороженными слоями:")
model_frozen.summary()

# Колбэки для улучшения обучения
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

# Обучение с заморозкой
print("Начало обучения с заморозкой...")
start_time = time.time()
history_frozen = model_frozen.fit(
    train_dataset,
    epochs=10,  # Увеличиваем эпохи
    validation_data=test_dataset,
    verbose=1,
    callbacks=callbacks
)
frozen_training_time = time.time() - start_time

# 4.2 ОБУЧЕНИЕ БЕЗ ЗАМОРОЗКИ
print("\n4.2 Обучение БЕЗ ЗАМОРОЗКИ (все слои обучаемые)...")

model_unfrozen, base_model_unfrozen = create_model_unfrozen()

start_time = time.time()
history_unfrozen = model_unfrozen.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    verbose=1,
    callbacks=callbacks
)
unfrozen_training_time = time.time() - start_time

# 5. СРАВНЕНИЕ РЕЗУЛЬТАТОВ
print("\n5. Сравнение качества и скорости обучения...")

# Финальная оценка моделей
print("\n=== ФИНАЛЬНАЯ ОЦЕНКА ===")

# Модель с заморозкой
test_loss_frozen, test_accuracy_frozen = model_frozen.evaluate(test_dataset, verbose=0)
print(f"Модель с ЗАМОРОЗКОЙ:")
print(f"  - Test Accuracy: {test_accuracy_frozen:.4f} ({test_accuracy_frozen*100:.1f}%)")
print(f"  - Test Loss: {test_loss_frozen:.4f}")
print(f"  - Время обучения: {frozen_training_time:.2f} сек")

# Модель без заморозки
test_loss_unfrozen, test_accuracy_unfrozen = model_unfrozen.evaluate(test_dataset, verbose=0)
print(f"\nМодель БЕЗ ЗАМОРОЗКИ:")
print(f"  - Test Accuracy: {test_accuracy_unfrozen:.4f} ({test_accuracy_unfrozen*100:.1f}%)")
print(f"  - Test Loss: {test_loss_unfrozen:.4f}")
print(f"  - Время обучения: {unfrozen_training_time:.2f} сек")

# Сравнение улучшения
accuracy_improvement = test_accuracy_unfrozen - test_accuracy_frozen
time_difference = unfrozen_training_time - frozen_training_time

print(f"\n=== СРАВНЕНИЕ ===")
print(f"Улучшение accuracy: {accuracy_improvement:.4f}")
print(f"Разница во времени: {time_difference:.2f} сек")

if accuracy_improvement > 0:
    improvement_percent = (accuracy_improvement / test_accuracy_frozen) * 100
    print(f"Процент улучшения: {improvement_percent:.1f}%")
else:
    print("Замороженная модель показала лучший результат!")

# 6. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
print("\n6. Визуализация результатов обучения...")

plt.figure(figsize=(15, 10))

# График точности
plt.subplot(2, 2, 1)
plt.plot(history_frozen.history['accuracy'], label='Frozen Training', linestyle='--', linewidth=2)
plt.plot(history_frozen.history['val_accuracy'], label='Frozen Validation', linestyle='--', linewidth=2)
plt.plot(history_unfrozen.history['accuracy'], label='Unfrozen Training', linewidth=2)
plt.plot(history_unfrozen.history['val_accuracy'], label='Unfrozen Validation', linewidth=2)
plt.title('Сравнение Accuracy (10% данных)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# График потерь
plt.subplot(2, 2, 2)
plt.plot(history_frozen.history['loss'], label='Frozen Training', linestyle='--', linewidth=2)
plt.plot(history_frozen.history['val_loss'], label='Frozen Validation', linestyle='--', linewidth=2)
plt.plot(history_unfrozen.history['loss'], label='Unfrozen Training', linewidth=2)
plt.plot(history_unfrozen.history['val_loss'], label='Unfrozen Validation', linewidth=2)
plt.title('Сравнение Loss (10% данных)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Сравнение времени
plt.subplot(2, 2, 3)
models = ['С заморозкой', 'Без заморозки']
times = [frozen_training_time, unfrozen_training_time]
bars = plt.bar(models, times, color=['lightblue', 'lightcoral'])
plt.title('Время обучения (10% данных)')
plt.ylabel('Секунды')
for bar, time_val in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{time_val:.1f}с', ha='center', va='bottom')
plt.grid(True, alpha=0.3)

# Сравнение точности
plt.subplot(2, 2, 4)
accuracies = [test_accuracy_frozen, test_accuracy_unfrozen]
bars = plt.bar(models, accuracies, color=['lightblue', 'lightcoral'])
plt.title('Финальная точность на тесте (10% данных)')
plt.ylabel('Accuracy')
for bar, accuracy in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.01, 
             f'{accuracy:.3f}', ha='center', va='bottom')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== ВЫВОДЫ (на 10% данных) ===")
print(f"1. ЗАМОРОЗКА: Accuracy = {test_accuracy_frozen*100:.1f}%, Время = {frozen_training_time:.1f}с")
print(f"2. БЕЗ ЗАМОРОЗКИ: Accuracy = {test_accuracy_unfrozen*100:.1f}%, Время = {unfrozen_training_time:.1f}с")
print(f"3. Эффективность: Заморозка в {unfrozen_training_time/frozen_training_time:.1f} раз быстрее")

print("\n=== ЗАДАНИЕ №1 ВЫПОЛНЕНО! ===")

## Часть 2 — Автоэнкодеры
**Методы:** convolutional autoencoder (2 слоя) vs improved model (3 слоя), MSE + PSNR.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print("=== ЗАДАНИЕ №2: АВТОЭНКОДЕРЫ ===")

# 1. Загрузка и подготовка данных MNIST
print("1. Загрузка данных MNIST...")
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Нормализация и reshaping
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

print(f"Размер тренировочной выборки: {x_train.shape}")
print(f"Размер тестовой выборки: {x_test.shape}")

# 2. Добавление шума
print("2. Добавление шума...")
def add_noise(images, noise_factor=0.5):
    noise = np.random.normal(loc=0.5, scale=0.5, size=images.shape)
    noisy_images = images + noise_factor * noise
    return np.clip(noisy_images, 0.0, 1.0)

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)

# 3. БАЗОВАЯ МОДЕЛЬ (из примера) - 2 слоя свертки/развертки
print("3. Создание базовой модели (2 слоя)...")

def create_basic_autoencoder():
    # Энкодер
    encoder_inputs = keras.Input(shape=(28, 28, 1), name='encoder_input')
    
    x = keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', strides=2)(encoder_inputs)
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', strides=2)(x)
    
    shape_before_flatten = x.shape
    x = keras.layers.Flatten()(x)
    latent = keras.layers.Dense(16, name='latent_vector')(x)
    
    encoder = keras.Model(encoder_inputs, latent, name='encoder')
    
    # Декодер
    latent_inputs = keras.Input(shape=(16,), name='decoder_input')
    x = keras.layers.Dense(shape_before_flatten[1] * shape_before_flatten[2] * shape_before_flatten[3])(latent_inputs)
    x = keras.layers.Reshape((shape_before_flatten[1], shape_before_flatten[2], shape_before_flatten[3]))(x)
    
    x = keras.layers.Conv2DTranspose(64, (3, 3), activation='relu', padding='same', strides=2)(x)
    x = keras.layers.Conv2DTranspose(32, (3, 3), activation='relu', padding='same', strides=2)(x)
    x = keras.layers.Conv2DTranspose(1, (3, 3), padding='same')(x)
    outputs = keras.layers.Activation('sigmoid', name='decoder_output')(x)
    
    decoder = keras.Model(latent_inputs, outputs, name='decoder')
    
    # Автоэнкодер
    autoencoder = keras.Model(encoder_inputs, decoder(encoder(encoder_inputs)), name='autoencoder_basic')
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder, encoder, decoder

# 4. АЛЬТЕРНАТИВНАЯ МОДЕЛЬ - 3 слоя свертки/развертки
print("4. Создание альтернативной модели (3 слоя)...")
def create_advanced_autoencoder():
    # Энкодер
    encoder_inputs = keras.Input(shape=(28, 28, 1), name='encoder_input')
    
    x = keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_inputs)
    x = keras.layers.MaxPooling2D((2, 2), padding='same')(x)
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2), padding='same')(x)
    x = keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    
    shape_before_flatten = x.shape
    x = keras.layers.Flatten()(x)
    latent = keras.layers.Dense(32, name='latent_vector')(x)  # Увеличили размерность латентного пространства
    
    encoder = keras.Model(encoder_inputs, latent, name='encoder_advanced')
    
    # Декодер
    latent_inputs = keras.Input(shape=(32,), name='decoder_input')
    x = keras.layers.Dense(shape_before_flatten[1] * shape_before_flatten[2] * shape_before_flatten[3])(latent_inputs)
    x = keras.layers.Reshape((shape_before_flatten[1], shape_before_flatten[2], shape_before_flatten[3]))(x)
    
    x = keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.UpSampling2D((2, 2))(x)
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.UpSampling2D((2, 2))(x)
    x = keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.Conv2D(1, (3, 3), padding='same')(x)
    outputs = keras.layers.Activation('sigmoid', name='decoder_output')(x)
    
    decoder = keras.Model(latent_inputs, outputs, name='decoder_advanced')
    
    # Автоэнкодер
    autoencoder = keras.Model(encoder_inputs, decoder(encoder(encoder_inputs)), name='autoencoder_advanced')
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder, encoder, decoder

# 5. Обучение моделей
print("5. Обучение моделей...")

# Базовая модель (2 слоя)
autoencoder_basic, encoder_basic, decoder_basic = create_basic_autoencoder()
print("Базовая модель (2 слоя):")
autoencoder_basic.summary()

history_basic = autoencoder_basic.fit(
    x_train_noisy, x_train,
    epochs=15,
    batch_size=128,
    validation_data=(x_test_noisy, x_test),
    verbose=1
)

# Альтернативная модель (3 слоя)
autoencoder_advanced, encoder_advanced, decoder_advanced = create_advanced_autoencoder()
print("\nАльтернативная модель (3 слоя):")
autoencoder_advanced.summary()

history_advanced = autoencoder_advanced.fit(
    x_train_noisy, x_train,
    epochs=15,
    batch_size=128,
    validation_data=(x_test_noisy, x_test),
    verbose=1
)

# 6. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ (пункты 2.1 и 2.2)
print("6. Визуализация результатов...")

# Предсказания обеих моделей
decoded_imgs_basic = autoencoder_basic.predict(x_test_noisy)
decoded_imgs_advanced = autoencoder_advanced.predict(x_test_noisy)

# Визуализация сравнения моделей
n = 10  # Количество примеров для отображения
plt.figure(figsize=(20, 8))

for i in range(n):
    # Оригинальные зашумленные
    ax = plt.subplot(4, n, i + 1)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    plt.title('Noisy')
    plt.axis('off')
    
    # Базовая модель (2 слоя)
    ax = plt.subplot(4, n, i + 1 + n)
    plt.imshow(decoded_imgs_basic[i].reshape(28, 28), cmap='gray')
    plt.title('Basic (2 layers)')
    plt.axis('off')
    
    # Альтернативная модель (3 слоя)
    ax = plt.subplot(4, n, i + 1 + 2*n)
    plt.imshow(decoded_imgs_advanced[i].reshape(28, 28), cmap='gray')
    plt.title('Advanced (3 layers)')
    plt.axis('off')
    
    # Оригинальные чистые
    ax = plt.subplot(4, n, i + 1 + 3*n)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.title('Original')
    plt.axis('off')

plt.suptitle('Сравнение моделей автоэнкодеров', fontsize=16)
plt.tight_layout()
plt.show()

# 7. СРАВНЕНИЕ КАЧЕСТВА (пункт 2.3)
print("7. Сравнение качества моделей...")

# Оценка моделей на тестовых данных
test_loss_basic = autoencoder_basic.evaluate(x_test_noisy, x_test, verbose=0)
test_loss_advanced = autoencoder_advanced.evaluate(x_test_noisy, x_test, verbose=0)
print("\n=== РЕЗУЛЬТАТЫ СРАВНЕНИЯ ===")
print(f"Базовая модель (2 слоя):")
print(f"  - Test Loss: {test_loss_basic:.4f}")
print(f"  - Параметры: {autoencoder_basic.count_params():,}")

print(f"\nАльтернативная модель (3 слоя):")
print(f"  - Test Loss: {test_loss_advanced:.4f}")
print(f"  - Параметры: {autoencoder_advanced.count_params():,}")

# Сравнение потери
if test_loss_advanced < test_loss_basic:
    improvement = ((test_loss_basic - test_loss_advanced) / test_loss_basic) * 100
    print(f"\n✅ Альтернативная модель лучше на {improvement:.1f}%")
else:
    improvement = ((test_loss_advanced - test_loss_basic) / test_loss_advanced) * 100
    print(f"\n✅ Базовая модель лучше на {improvement:.1f}%")

# 8. Графики обучения
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_basic.history['loss'], label='Basic Training')
plt.plot(history_basic.history['val_loss'], label='Basic Validation')
plt.plot(history_advanced.history['loss'], label='Advanced Training')
plt.plot(history_advanced.history['val_loss'], label='Advanced Validation')
plt.title('Loss during Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
# PSNR (Peak Signal-to-Noise Ratio) - метрика качества изображений
def calculate_psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return 100
    psnr = 20 * np.log10(1.0 / np.sqrt(mse))
    return psnr

psnr_basic = calculate_psnr(x_test, decoded_imgs_basic)
psnr_advanced = calculate_psnr(x_test, decoded_imgs_advanced)

models = ['Basic (2 layers)', 'Advanced (3 layers)']
psnr_values = [psnr_basic, psnr_advanced]

plt.bar(models, psnr_values, color=['blue', 'orange'])
plt.title('PSNR Comparison (Higher is Better)')
plt.ylabel('PSNR (dB)')

plt.tight_layout()
plt.show()

# 9. Сохранение моделей
autoencoder_basic.save('autoencoder_basic_2layers.h5')
autoencoder_advanced.save('autoencoder_advanced_3layers.h5')
print("\nМодели сохранены:")
print("- autoencoder_basic_2layers.h5")
print("- autoencoder_advanced_3layers.h5")

print("\n=== ЗАДАНИЕ №2 ВЫПОЛНЕНО! ===")